In [3]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix

In [5]:
from pathlib import Path

base_dir = Path.cwd()
candidate_files = [
    base_dir / 'heart_dataset.csv',
    base_dir / 'heart_failure_clinical_records_dataset.csv',
    base_dir / 'heart.csv',
    base_dir / 'dataset' / 'heart_dataset.csv',
    base_dir / 'data' / 'heart_dataset.csv',
]

input_path = next((p for p in candidate_files if p.exists()), None)
if input_path is not None:
    data = pd.read_csv(input_path)
    print(f'Dataset cargado desde archivo local: {input_path}')
else:
    fallback_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00519/heart_failure_clinical_records_dataset.csv'
    data = pd.read_csv(fallback_url)
    print(f'Dataset cargado desde URL: {fallback_url}')

df = data.copy()
data.head(10)

Dataset cargado desde URL: https://archive.ics.uci.edu/ml/machine-learning-databases/00519/heart_failure_clinical_records_dataset.csv


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1
5,90.0,1,47,0,40,1,204000.00,2.1,132,1,1,8,1
6,75.0,1,246,0,15,0,127000.00,1.2,137,1,0,10,1
7,60.0,1,315,1,60,0,454000.00,1.1,131,1,1,10,1
8,65.0,0,157,0,65,0,263358.03,1.5,138,0,0,10,1
9,80.0,1,123,0,35,1,388000.00,9.4,133,1,1,10,1


In [6]:
data.describe()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.00000,299.000000,299.000000,299.00000,299.000000,299.00000
mean,60.833893,0.431438,581.839465,0.418060,38.083612,0.351171,263358.029264,1.39388,136.625418,0.648829,0.32107,130.260870,0.32107
std,11.894809,0.496107,970.287881,0.494067,11.834841,0.478136,97804.236869,1.03451,4.412477,0.478136,0.46767,77.614208,0.46767
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.50000,113.000000,0.000000,0.00000,4.000000,0.00000
25%,51.000000,0.000000,116.500000,0.000000,30.000000,0.000000,212500.000000,0.90000,134.000000,0.000000,0.00000,73.000000,0.00000
50%,60.000000,0.000000,250.000000,0.000000,38.000000,0.000000,262000.000000,1.10000,137.000000,1.000000,0.00000,115.000000,0.00000
75%,70.000000,1.000000,582.000000,1.000000,45.000000,1.000000,303500.000000,1.40000,140.000000,1.000000,1.00000,203.000000,1.00000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.40000,148.000000,1.000000,1.00000,285.000000,1.00000


In [7]:
inp_data = data.drop(columns=['DEATH_EVENT'])
out_data = data['DEATH_EVENT']

scaler = StandardScaler()
inp_data = scaler.fit_transform(inp_data)

X_train, X_test, y_train, y_test = train_test_split(
    inp_data, out_data, test_size=0.2, random_state=42, stratify=out_data
)

In [8]:
clf_base = SVC()
clf_base.fit(X_train, y_train)

y_pred_base = clf_base.predict(X_test)

base_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_base),
    'Precision': precision_score(y_test, y_pred_base, zero_division=0),
    'Recall': recall_score(y_test, y_pred_base, zero_division=0),
    'F1': f1_score(y_test, y_pred_base, zero_division=0),
}

print('Metricas modelo base:')
for k, v in base_metrics.items():
    print(f'{k}: {v:.4f}')

Metricas modelo base:
Accuracy: 0.7667
Precision: 0.7273
Recall: 0.4211
F1: 0.5333


In [9]:
# Busqueda de mejores hiperparametros
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
c_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
gammas = [0.001, 0.01, 0.1, 1, 10]

param_grid = {'kernel': kernels, 'C': c_values, 'gamma': gammas}

grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)
grid.fit(X_train, y_train)

print('Mejores parametros encontrados:')
print(grid.best_params_)
print(f"Mejor F1 en validacion cruzada: {grid.best_score_:.4f}")

Mejores parametros encontrados:
{'C': 1, 'gamma': 0.001, 'kernel': 'linear'}
Mejor F1 en validacion cruzada: 0.7542


In [10]:
best_params = grid.best_params_
clf_tuned = SVC(**best_params)
clf_tuned.fit(X_train, y_train)

y_pred_tuned = clf_tuned.predict(X_test)

tuned_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned, zero_division=0),
    'Recall': recall_score(y_test, y_pred_tuned, zero_division=0),
    'F1': f1_score(y_test, y_pred_tuned, zero_division=0),
}

comparison_df = pd.DataFrame([base_metrics, tuned_metrics], index=['Base', 'Tuned'])
print('Comparacion de metricas (Base vs Tuned):')
display(comparison_df.round(4))

print('\nMatriz de confusion (modelo tuned):')
print(confusion_matrix(y_test, y_pred_tuned))

Comparacion de metricas (Base vs Tuned):


,Accuracy,Precision,Recall,F1
Base,0.7667,0.7273,0.4211,0.5333
Tuned,0.7833,0.8000,0.4211,0.5517



Matriz de confusion (modelo tuned):
[[39  2]
 [11  8]]
